# Classical ML Pipeline — Training (SVM / KNN / Logistic Regression)

Uses the same 1280-dim MobileNetV2 `GlobalAveragePooling2D` embeddings cached by `xgb_train.ipynb`.
No GPU or image loading required — all three models train directly on `features/features_*.npy`.

**Models trained:**
- **SVM** — Linear SVM + Platt calibration (`LinearSVC` + `CalibratedClassifierCV`). `StandardScaler` applied first.
- **KNN** — K-Nearest Neighbours (k=11, ball-tree). `StandardScaler` applied first.
- **LR** — Logistic Regression (L2, C=1). `StandardScaler` applied first.

Stage 1: 3-class structure classifier (Decks / Pavements / Walls)  
Stage 2: binary defect classifier per structure, `class_weight='balanced'`  
Threshold tuning: same sweep logic as CNN / XGBoost pipelines for a fair comparison.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import recall_score, f1_score

STRUCTURES      = ['Decks', 'Pavements', 'Walls']
THRESHOLD_SWEEP = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
MIN_F1          = 0.60
SEED            = 42

np.random.seed(SEED)
print('All imports OK')

## 1. Load Embeddings & Splits

In [ ]:
df_train = pd.read_csv('splits/train_split.csv')
df_val   = pd.read_csv('splits/val_split.csv')
df_test  = pd.read_csv('splits/test_split.csv')

X_train = np.load('features/features_train.npy')
X_val   = np.load('features/features_val.npy')
X_test  = np.load('features/features_test.npy')

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 2. Model Definitions

All models are wrapped in a `Pipeline` with `StandardScaler` first — important for SVM and LR
since CNN embeddings are not unit-normalised. KNN also benefits from scaling.

`LinearSVC` is wrapped with `CalibratedClassifierCV` (Platt scaling) to produce
calibrated probabilities needed for threshold tuning and ROC-AUC.

In [ ]:
def make_svm(class_weight=None):
    base = Pipeline([
        ('scaler', StandardScaler()),
        ('svm',    LinearSVC(class_weight=class_weight, max_iter=5000, C=1.0, random_state=SEED)),
    ])
    return CalibratedClassifierCV(base, cv=3)

def make_knn():
    # KNN is O(n) at query time in high dimensions — subsample training set to keep it fast.
    # 6000 stratified samples is representative; more data does not improve KNN past a point.
    return Pipeline([
        ('scaler', StandardScaler()),
        ('knn',    KNeighborsClassifier(n_neighbors=11, algorithm='brute',
                                        metric='cosine', n_jobs=-1)),
    ])

def make_lr(class_weight=None):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(class_weight=class_weight, max_iter=1000,
                                      C=1.0, random_state=SEED, n_jobs=-1)),
    ])

KNN_MAX_TRAIN = 6000  # cap KNN training set to avoid O(n*d) query explosion

def subsample(X, y, n, seed=SEED):
    """Stratified subsample keeping class proportions."""
    rng = np.random.default_rng(seed)
    idx = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        take = max(1, int(n * len(cls_idx) / len(y)))
        idx.extend(rng.choice(cls_idx, min(take, len(cls_idx)), replace=False))
    return X[idx], y[idx]

MODELS = {
    'svm': {'factory': make_svm, 'label': 'SVM (Linear + Calibrated)', 'knn': False},
    'knn': {'factory': make_knn, 'label': f'KNN (k=11, cosine, n≤{KNN_MAX_TRAIN})', 'knn': True},
    'lr':  {'factory': make_lr,  'label': 'Logistic Regression', 'knn': False},
}
print('Model factories defined')

## 3. Stage 1 — Structure Classifiers

In [ ]:
y_struct_train = df_train['structure_idx'].values
y_struct_val   = df_val['structure_idx'].values

Path('models/classical').mkdir(parents=True, exist_ok=True)

for key, info in MODELS.items():
    print(f'Training Stage 1 — {info["label"]} ...', end=' ', flush=True)
    cw = None
    Xtr, ytr = X_train, y_struct_train
    if info['knn']:
        Xtr, ytr = subsample(X_train, y_struct_train, KNN_MAX_TRAIN)
        model = info['factory']()
    else:
        model = info['factory'](class_weight=cw)
    model.fit(Xtr, ytr)
    val_acc = (model.predict(X_val) == y_struct_val).mean()
    print(f'val_acc={val_acc:.3f}  (trained on {len(ytr):,} samples)')
    fname = f'models/classical/{key}_structure_model.pkl'
    with open(fname, 'wb') as f:
        pickle.dump(model, f)
    print(f'  Saved {fname}')

## 4. Stage 2 — Per-Structure Defect Classifiers

In [ ]:
all_thresholds = {key: {} for key in MODELS}

for key, info in MODELS.items():
    print(f'\n{"="*60}')
    print(f'  {info["label"]} — Stage 2 Defect Classifiers')
    print(f'{"="*60}')

    for struct in STRUCTURES:
        mask_tr = (df_train['structure'] == struct).values
        mask_vl = (df_val['structure']   == struct).values

        X_tr = X_train[mask_tr]
        y_tr = df_train.loc[mask_tr, 'defect_idx'].values
        X_vl = X_val[mask_vl]
        y_vl = df_val.loc[mask_vl, 'defect_idx'].values

        if info['knn']:
            Xtr_fit, ytr_fit = subsample(X_tr, y_tr, KNN_MAX_TRAIN)
            model = info['factory']()
        else:
            Xtr_fit, ytr_fit = X_tr, y_tr
            model = info['factory'](class_weight='balanced')

        n_pos = int((ytr_fit == 1).sum())
        n_neg = int((ytr_fit == 0).sum())
        print(f'\n  {struct}: {n_pos} defect / {n_neg} no_defect  (n={len(ytr_fit):,})', end=' ')
        model.fit(Xtr_fit, ytr_fit)
        print('trained', end=' ')

        fname = f'models/classical/{key}_defect_{struct.lower()}.pkl'
        with open(fname, 'wb') as f:
            pickle.dump(model, f)
        print(f'→ saved')

        val_probs = model.predict_proba(X_vl)[:, 1]
        best_thresh, found = 0.50, False
        sweep_rows = []
        for t in THRESHOLD_SWEEP:
            preds = (val_probs >= t).astype(int)
            rec = recall_score(y_vl, preds, zero_division=0)
            f1  = f1_score(y_vl, preds, zero_division=0)
            sweep_rows.append(f'    t={t:.2f}  recall={rec:.3f}  f1={f1:.3f}')
            if f1 >= MIN_F1 and not found:
                best_thresh, found = t, True
        if not found:
            f1s = [f1_score(y_vl, (val_probs >= t).astype(int), zero_division=0)
                   for t in THRESHOLD_SWEEP]
            best_thresh = THRESHOLD_SWEEP[int(np.argmax(f1s))]
            print(f'  WARNING: no threshold reached F1>={MIN_F1}; using best-F1 threshold.')
        for row in sweep_rows:
            print(row)
        print(f'  → chosen threshold for {struct}: {best_thresh}')
        all_thresholds[key][struct] = best_thresh

with open('models/classical/classical_thresholds.json', 'w') as f:
    json.dump(all_thresholds, f, indent=2)
print('\nSaved models/classical/classical_thresholds.json')
print(all_thresholds)

## 5. Summary

In [ ]:
print('=== Classical ML Training Complete ===')
print()
expected = (
    [f'models/classical/{k}_structure_model.pkl' for k in MODELS] +
    [f'models/classical/{k}_defect_{s.lower()}.pkl' for k in MODELS for s in STRUCTURES] +
    ['models/classical/classical_thresholds.json']
)
for fname in expected:
    status = 'OK' if Path(fname).exists() else 'MISSING'
    print(f'  [{status}] {fname}')